In [8]:
import pandas as pd
secondary_metabolites = pd.read_html("https://www.genome.jp/kegg/tables/br08011.html", header=0)

In [9]:
secondary_metabolites[0]

,Map,Pathway,Module,KEGG ID,Name,Remark
0,00902,Geraniol biosynthesis,NaN,C01500,Geraniol,Monoterpenoid
1,00902,Myrcene biosynthesis,NaN,C06074,Myrcene,Monoterpenoid
2,00902,Limonene biosynthesis,NaN,C00521 C06099,(S)-Limonene (R)-Limonene,Monoterpenoid
3,00902,Menthol biosynthesis,NaN,C00400,Menthol,Monoterpenoid
4,00902,Carvone biosynthesis,NaN,C01767 C11383,(-)-Carvone (+)-Carvone,Monoterpenoid
...,...,...,...,...,...,...
148,00997,Aerobactin biosynthesis,M00918,C05554,Aerobactin,Siderophore
149,00997,Cyclooctatin biosynthesis,M00921,C21979,Cyclooctatin,"Diterpenoid, lysophospholipase inhibitor"
150,00997,Lovastatin biosynthesis,M00893,C21130,Lovastatin acid,Fungal polyketide
151,00997,Grixazone biosynthesis,M00905,C20799,Grixazone B,"Grixazone, yellow pigment"


In [10]:
import io
from Bio.KEGG import REST as kegg_api


def get_ec_in_pathway(pathway):
    res = []
    try:
        result = kegg_api.kegg_link("enzyme", f"path:{pathway}").read()
        try:
            result_enzymes = pd.read_table(io.StringIO(result), header=None)
            for enzyme in result_enzymes.iloc[:, 1]:
                res.append(enzyme.replace("ec:", ""))

        except:
            pass
    except:
        print(f"{pathway}")
        return res

    return res

In [11]:
maps = secondary_metabolites[0].loc[:, ["Map", "Remark"]]
maps

,Map,Remark
0,00902,Monoterpenoid
1,00902,Monoterpenoid
2,00902,Monoterpenoid
3,00902,Monoterpenoid
4,00902,Monoterpenoid
...,...,...
148,00997,Siderophore
149,00997,"Diterpenoid, lysophospholipase inhibitor"
150,00997,Fungal polyketide
151,00997,"Grixazone, yellow pigment"


In [12]:
maps_enzymes = {}
for i in range(len(maps)):
    map_ = maps.iloc[i, 0]
    description = maps.iloc[i, 1]

    maps_ = map_.split(" ")
    for map_ in maps_:
        map_clean = f"map{map_}"
        if map_clean not in maps_enzymes:
            maps_enzymes[map_clean] = get_ec_in_pathway(map_clean)


map01057


In [20]:
maps_enzymes["map00253"] = get_ec_in_pathway("map00253")

In [13]:
len(maps_enzymes)

43

In [28]:
import json

with open("secondary_metabolism_enzymes_plantcyc.json", "r") as f:
    maps_enzymes_plant_cyc = json.load(f)

In [29]:
sm_ec_numbers = []
for ec in maps_enzymes_plant_cyc.values():
    sm_ec_numbers.extend(ec)

In [30]:
sm_ec_numbers = list(set(sm_ec_numbers))

In [31]:
len(sm_ec_numbers)

419

In [17]:
for ec in maps_enzymes_plant_cyc.values():
    sm_ec_numbers.extend(ec)

In [32]:
len(sm_ec_numbers)

419

In [33]:
import pandas as pd

ec_plants = pd.read_csv("data/swiss_prot_ec_plants.csv")
ids_sm = []
for i, ec_numbers in enumerate(ec_plants.EC):
    ec = ec_numbers.split(";")
    for ec_ in ec:
        if ec_ in sm_ec_numbers:
            ids_sm.append(i)
            break

In [34]:
ids_sm

[0,
 1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 21,
 22,
 23,
 51,
 59,
 60,
 61,
 156,
 158,
 159,
 165,
 166,
 167,
 168,
 169,
 170,
 171,
 172,
 173,
 174,
 175,
 176,
 177,
 178,
 179,
 180,
 181,
 182,
 184,
 189,
 210,
 224,
 225,
 237,
 239,
 240,
 241,
 268,
 270,
 271,
 300,
 301,
 323,
 324,
 325,
 326,
 327,
 363,
 398,
 399,
 400,
 401,
 421,
 422,
 423,
 444,
 445,
 446,
 447,
 483,
 484,
 485,
 518,
 555,
 560,
 561,
 568,
 569,
 572,
 579,
 582,
 585,
 592,
 593,
 622,
 623,
 624,
 625,
 629,
 630,
 631,
 632,
 633,
 634,
 635,
 636,
 637,
 638,
 639,
 640,
 641,
 642,
 643,
 644,
 648,
 662,
 663,
 665,
 666,
 667,
 668,
 669,
 670,
 674,
 708,
 709,
 710,
 711,
 712,
 713,
 714,
 715,
 716,
 717,
 718,
 719,
 720,
 721,
 748,
 831,
 832,
 834,
 836,
 837,
 1013,
 1262,
 1263,
 1285,
 1311,
 1312,
 1313,
 1352,
 1361,
 1377,
 1378,
 1379,
 1381,
 1382,
 1383,
 1385,
 1386,
 1387,
 1388,
 1389,
 1390,
 1391,
 1392,
 1393,
 1448,
 14

In [35]:
ec_plants_sm = ec_plants.iloc[ids_sm, :]

In [36]:
ec_plants_sm.to_csv("swiss_prot_ec_plants_sm.csv", index=False)

In [1]:
import json

with open("maps_secondary_metabolism_ec_numbers.json", "r") as f:
    maps_enzymes = json.load(f)

sm_ec_numbers = []

for ec in maps_enzymes.values():
    sm_ec_numbers.extend(ec)

In [2]:
import pandas as pd
ec_plants = pd.read_csv("swiss_prot_ec_plants.csv")
ids_sm = []
for i, ec_numbers in enumerate(ec_plants.EC):
    ec = ec_numbers.split(";")
    for ec_ in ec:
        if ec_ in sm_ec_numbers:
            ids_sm.append(i)
            break

In [5]:
ec_plants_sm = ec_plants.iloc[ids_sm, :]
ec_plants_sm.head()

,accession,name,sequence,EC,lineage,species,taxonomy_id,1,2,3,...,7.4.2.5,7.4.2.8,7.5.2.11,7.6.2.1,7.6.2.11,7.6.2.13,7.6.2.2,7.6.2.3,7.6.2.5,7.6.2.8
11,Q9MBC1,3AT_PERFR,VIETCRVGPPPDSVAEQSVPLTFFDMTWLHFHPMLQLLFYEFPCSK...,2.3.1.215,Eukaryota;Viridiplantae;Streptophyta;Embryophy...,Perilla frutescens,48386,0,1,0,...,0,0,0,0,0,0,0,0,0,0
26,Q41247,AL7A1_BRANA,MGSASKEYEFLSEIGLSSSHNLGNYVGGKWLGNGPLVSTLNPANNQ...,1.2.1.3,Eukaryota;Viridiplantae;Streptophyta;Embryophy...,Brassica napus,3708,1,0,0,...,0,0,0,0,0,0,0,0,0,0
30,P49252,AMO_LENCU,KFALFSVLTLLSFHAVFSFTPLHTQHPLDPITKEEFLAVQTIVQNK...,1.4.3.21,Eukaryota;Viridiplantae;Streptophyta;Embryophy...,Lens culinaris,3864,1,0,0,...,0,0,0,0,0,0,0,0,0,0
40,A2XNK3,ASA1_ORYSI,MASLVLSLRIAPSTPPLGLGGGRFRGRRGAVACRAATFQQLDAVAV...,4.1.3.27,Eukaryota;Viridiplantae;Streptophyta;Embryophy...,Oryza sativa subsp. indica,39946,0,0,0,...,0,0,0,0,0,0,0,0,0,0
78,A0A0S2IHL6,BASM1_KALSE,MWKLKIAEGDKNDPYLYSTNNFVGRQTWEFDPDYVGSPGELEEVEE...,5.4.99.39,Eukaryota;Viridiplantae;Streptophyta;Embryophy...,Kalopanax septemlobus,228393,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [22]:
import json

# save json file
with open("maps_secondary_metabolism_ec_numbers.json", "w") as f:
    json.dump(maps_enzymes, f)

In [37]:
ec_plants = pd.read_csv("./data/trembl_prot_ec_plants.csv")
ids_sm = []
for i, ec_numbers in enumerate(ec_plants.EC):
    ec = ec_numbers.split(";")
    for ec_ in ec:
        if ec_ in sm_ec_numbers:
            ids_sm.append(i)
            break

In [38]:
ec_plants_sm = ec_plants.iloc[ids_sm, :]

In [39]:
ec_plants_sm.shape

(47600, 7)

In [40]:
ec_plants_sm.to_csv("./trembl_prot_ec_plants_sm.csv", index=False)